# 4.2 Demand Prediction

In this notebook we implement Neural Networks (NNs) to predict taxi trip demand in chicago. Additionally, we compare the performances of the NNs across different complexity levels.

For NNs there are 2 "main" complexity interpretations:
- depth: number of hidden layers
- width: number of nodes per layer
- (more complex activation function) - maybe as an extra
- (more complex optimizer) - maybe as an extra

So we decide to test 3 different NN structures:

__baseline model__:
- hidden layers: 2
- nodes per layer: 64

__wider model__:
- hidden layers: 2
- nodes per layer: 128

__deeper model__:
- hidden layers: 4
- nodes per layer: 64

For better comparison, we will test all three architectures with the same shared configurations.

In [15]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
# import matplotlib.pyplot as plt

# modeling
import copy
import random
import types

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# model visualization
from torchinfo import summary

# reset working dir
import os
from pathlib import Path


In [16]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/anthony/Documents/Dokumente – MacBook Pro von Anthony/UNI/AAA/AAA_TA_2026


In [17]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load data                               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data = pd.read_parquet("data/aggregated/hexagon/demand_hex_1h_medium.parquet")

In [18]:
data.head()

,time_bucket,bucket_index,pickup_h3_res7,area_type,trip_count,active_taxis,avg_idle_time,avg_trip_duration,avg_trip_distance,avg_fare,...,dist_to_nearest_train_station_km,dist_to_nearest_stadium_km,train_station_per_km2,restaurants_per_km2,bars_and_clubs_per_km2,hotels_per_km2,hospitals_per_km2,universities_per_km2,attractions_per_km2,poi_density_total_per_km2
0,2025-01-01,482136,872664190ffffff,residential,0,0,0.0,0.0,0.0,0.0,...,3.463389,5.191083,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000
1,2025-01-01,482136,872664191ffffff,residential,0,0,0.0,0.0,0.0,0.0,...,1.873244,4.149252,0.0,0.000000,0.385055,0.0,0.0,0.0,0.0,0.385055
2,2025-01-01,482136,872664192ffffff,residential,0,0,0.0,0.0,0.0,0.0,...,2.889357,4.325940,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000
3,2025-01-01,482136,872664193ffffff,residential,0,0,0.0,0.0,0.0,0.0,...,1.852723,6.519979,0.0,0.962827,0.385131,0.0,0.0,0.0,0.0,1.347958
4,2025-01-01,482136,872664194ffffff,residential,0,0,0.0,0.0,0.0,0.0,...,3.220069,4.757978,0.0,1.346856,0.577224,0.0,0.0,0.0,0.0,1.924079


In [19]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Shared Configs                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# --- optimization ---
OPTIMIZER          = "adam"
LEARNING_RATE      = 1e-5
BATCH_SIZE         = 256
MAX_EPOCHS         = 100

# --- lr search ---
LR_CANDIDATES      = [1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3]
LR_SEARCH_SEEDS    = (0, 1, 2)

# --- loss / output ---
LOSS               = "poisson_nll"     # demand is count data
OUTPUT_UNITS       = 1
OUTPUT_ACTIVATION  = "softplus"        # non-negative expected count

# --- layer defaults ---
HIDDEN_ACTIVATION  = "relu"
WEIGHT_INIT        = "he_normal"

# --- regularization (off by default to isolate the complexity effect) ---
DROPOUT            = 0.0
WEIGHT_DECAY       = 0.0

# --- early stopping ---
EARLY_STOPPING     = True
MONITOR            = "val_loss"
PATIENCE           = 10

# --- data handling ---
SPLIT              = "random"
SCALER_FIT_ON      = "train_only"

# --- hex embedding ---
HEX_EMBED_DIM      = 16               # learned embedding dim per hexagon

# --- device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- reproducibility ---
SEEDS              = (0, 1, 2, 3, 4)   # run each model across all seeds; report mean +/- std

# --- architectures (the ONLY thing that varies across the 3 models) ---
# format: (n_hidden_layers, width)
ARCH_BASELINE      = (2, 64)
ARCH_WIDER         = (2, 128)          # depth fixed, width up
ARCH_DEEPER        = (4, 64)           # width fixed, depth up
# expose as module-like object so model code can use config.XXX
config = types.SimpleNamespace(
    LEARNING_RATE  = LEARNING_RATE,
    WEIGHT_DECAY   = WEIGHT_DECAY,
    BATCH_SIZE     = BATCH_SIZE,
    MAX_EPOCHS     = MAX_EPOCHS,
    EARLY_STOPPING = EARLY_STOPPING,
    PATIENCE       = PATIENCE,
    OUTPUT_UNITS   = OUTPUT_UNITS,
    HEX_EMBED_DIM  = HEX_EMBED_DIM,
)

ARCH_NAMES = {ARCH_BASELINE: "baseline", ARCH_WIDER: "wide", ARCH_DEEPER: "deep"}

# per-arch best LRs from LR search; falls back to config.LEARNING_RATE if not set
BEST_LR = {}


In [20]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Model + Training Utilities              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

class DemandBaseline(nn.Module):
    def __init__(self, input_dim, n_layers, width, n_hex, embed_dim):
        super().__init__()
        self.hex_embed = nn.Embedding(n_hex, embed_dim)
        nn.init.normal_(self.hex_embed.weight, std=0.01)

        layers = []
        in_dim = input_dim + embed_dim
        for _ in range(n_layers):
            linear = nn.Linear(in_dim, width)
            nn.init.kaiming_normal_(linear.weight, nonlinearity="relu")
            nn.init.zeros_(linear.bias)
            layers += [linear, nn.ReLU()]
            in_dim = width
        out = nn.Linear(in_dim, config.OUTPUT_UNITS)
        nn.init.kaiming_normal_(out.weight, nonlinearity="relu")
        nn.init.zeros_(out.bias)
        layers += [out, nn.Softplus()]
        self.net = nn.Sequential(*layers)

    def forward(self, x, hex_idx):
        emb = self.hex_embed(hex_idx)          # (batch, embed_dim)
        x   = torch.cat([x, emb], dim=-1)      # (batch, input_dim + embed_dim)
        return self.net(x).squeeze(-1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_model(arch, X_train, y_train, X_val, y_val,
                hex_train, hex_val,
                seed=0, device="cpu", verbose=True, lr=None):
    set_seed(seed)
    n_layers, width = arch
    model = DemandBaseline(
        X_train.shape[1], n_layers, width, N_HEX, config.HEX_EMBED_DIM
    ).to(device)

    loss_fn   = nn.PoissonNLLLoss(log_input=False, full=False)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr if lr is not None else config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY,
    )

    train_dl = DataLoader(
        TensorDataset(
            torch.as_tensor(X_train,   dtype=torch.float32),
            torch.as_tensor(hex_train, dtype=torch.long),
            torch.as_tensor(y_train,   dtype=torch.float32),
        ),
        batch_size=config.BATCH_SIZE,
        shuffle=True,
    )
    X_val_t   = torch.as_tensor(X_val,   dtype=torch.float32).to(device)
    hex_val_t = torch.as_tensor(hex_val, dtype=torch.long).to(device)
    y_val_t   = torch.as_tensor(y_val,   dtype=torch.float32).to(device)

    best_val, best_state, wait = float("inf"), None, 0
    epoch_width = len(str(config.MAX_EPOCHS))

    for epoch in range(config.MAX_EPOCHS):
        # ── train ──────────────────────────────────────────────────────────────
        model.train()
        running_loss, n_batches = 0.0, 0
        for xb, hb, yb in train_dl:
            xb, hb, yb = xb.to(device), hb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb, hb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches    += 1
        train_loss = running_loss / n_batches

        # ── validate ───────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t, hex_val_t), y_val_t).item()

        # ── early stopping ─────────────────────────────────────────────────────
        improved = val_loss < best_val
        if improved:
            best_val, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1

        if verbose:
            marker = " *" if improved else f" (no improvement {wait}/{config.PATIENCE})"
            print(f"  epoch {epoch+1:{epoch_width}d}/{config.MAX_EPOCHS}"
                  f"  train={train_loss:.4f}  val={val_loss:.4f}{marker}")

        if config.EARLY_STOPPING and wait >= config.PATIENCE:
            if verbose:
                print(f"  early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    return model, best_val


def run_lr_search(arch, X_train, y_train, X_val, y_val,
                  hex_train, hex_val,
                  device="cpu", candidates=None, seeds=None):
    candidates  = candidates or LR_CANDIDATES
    seeds       = seeds      or LR_SEARCH_SEEDS
    arch_label  = ARCH_NAMES.get(arch, str(arch))
    n_layers, width = arch

    results = {}
    print(f"LR Search — {arch_label} ({n_layers} hidden layers, {width} units)")
    print(f"{'LR':>10}  {'mean val loss':>14}  {'std':>8}")
    print("-" * 38)

    for lr in candidates:
        val_losses = []
        for seed in seeds:
            _, val_loss = train_model(
                arch, X_train, y_train, X_val, y_val,
                hex_train, hex_val,
                seed=seed, device=device, verbose=False, lr=lr,
            )
            val_losses.append(val_loss)
        mean_loss = np.mean(val_losses)
        std_loss  = np.std(val_losses)
        results[lr] = {"mean": mean_loss, "std": std_loss}
        print(f"{lr:>10.0e}  {mean_loss:>14.4f}  {std_loss:>8.4f}")

    best_lr = min(results, key=lambda lr: results[lr]["mean"])
    print(f"\nBest LR : {best_lr:.0e}  "
          f"(val loss = {results[best_lr]['mean']:.4f} "
          f"± {results[best_lr]['std']:.4f})")

    BEST_LR[arch] = best_lr
    print(f"BEST_LR[{arch_label}] set to {best_lr:.0e}")
    return best_lr


## Model Architecture — Data Flow

```
One row of raw data
┌─────────────────┬──────────────────────────────┬─────────────┐
│ pickup_h3_res7  │  is_weekend, temp, rain, ...  │ trip_count  │
│ '872664190fff'  │  0,  12.3,  0.0,  ...        │      5      │
└────────┬────────┴───────────────┬───────────────┴──────┬──────┘
         │                       │                       │
         ▼                       ▼                       ▼
    hex_vocab              StandardScaler             target y
  '872664190fff'          [0.0, -0.3, ...]              5.0
       → 42
         │                       │
         ▼                       │
  Embedding table                │
  (600 hexagons × 16 dims)       │
  row 42: [0.12, -0.31, ...]     │
     (16 dims, learned)          │ (29 dims, scaled)
         │                       │
         └───────────┬───────────┘
                     ▼
               torch.cat(...)
          [0.12, -0.31, ..., 0.0, -0.3, ...]
                  (16 + 29 = 45 dims)
                     │
                     ▼
           ┌─────────────────────┐
           │  Linear(45 → 64)    │
           │  ReLU               │
           │  Linear(64 → 64)    │  ← baseline / deeper adds more blocks
           │  ReLU               │
           │  Linear(64 → 1)     │
           │  Softplus           │  ← keeps output ≥ 0 (trip count)
           └─────────────────────┘
                     │
                     ▼
              ŷ  (predicted trip count)
```

The embedding table starts with near-zero weights and is updated by backprop
alongside all other parameters — the network learns which hexagons are similar.


## Train / Val / Test Split

We split rows randomly into **70 % train / 15 % val / 15 % test** using
`train_test_split` with a fixed random state for reproducibility.

| Split | Share |
|-------|-------|
| Train | ~70 % |
| Val   | ~15 % |
| Test  | ~15 % |


In [21]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Train / Val / Test Split                #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data['time_bucket'] = pd.to_datetime(data['time_bucket'], format='mixed')

# 70 / 15 / 15 random split
train_data, temp_data = train_test_split(data, test_size=0.30, random_state=42)
val_data,   test_data = train_test_split(temp_data, test_size=0.50, random_state=42)

train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)
test_data  = test_data.reset_index(drop=True)

print(f"Train : {len(train_data):>9,} rows")
print(f"Val   : {len(val_data):>9,} rows")
print(f"Test  : {len(test_data):>9,} rows")


Train :   981,120 rows
Val   :   210,240 rows
Test  :   210,240 rows


## Feature Preparation

Drop leakage columns (trip-derived aggregates from the same time-bucket),
ID/index columns, categorical columns not yet encoded, and the target.  
Fit a `StandardScaler` on the **training set only** to avoid leakage into val/test.


In [22]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Feature Preparation                     #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# trip-derived stats (same time-bucket → leakage) + categoricals not yet encoded
LEAKAGE_COLS = [
    'active_taxis', 'avg_idle_time', 'avg_trip_duration', 'avg_trip_distance',
    'avg_fare', 'avg_trip_total', 'avg_tip', 'tip_rate', 'share_cash_payment',
    'area_type', 'season',
]

# raw cyclic integers replaced by sin/cos encodings
RAW_CYCLIC_COLS = ['month', 'hour_of_day', 'day_of_week']

ID_COLS    = ['time_bucket', 'bucket_index', 'pickup_h3_res7']
TARGET_COL = 'trip_count'

FEATURE_COLS = [
    c for c in train_data.columns
    if c not in LEAKAGE_COLS + RAW_CYCLIC_COLS + ID_COLS + [TARGET_COL]
]
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_data[FEATURE_COLS].values)
X_val   = scaler.transform(val_data[FEATURE_COLS].values)
X_test  = scaler.transform(test_data[FEATURE_COLS].values)

y_train = train_data[TARGET_COL].values.astype(float)
y_val   = val_data[TARGET_COL].values.astype(float)
y_test  = test_data[TARGET_COL].values.astype(float)

# hex embedding indices — vocabulary built from training set
hex_vocab = {h: i for i, h in enumerate(sorted(train_data['pickup_h3_res7'].unique()))}
N_HEX     = len(hex_vocab)

hex_train = train_data['pickup_h3_res7'].map(hex_vocab).values
hex_val   = val_data['pickup_h3_res7'].map(hex_vocab).values
hex_test  = test_data['pickup_h3_res7'].map(hex_vocab).values

print(f"\nX_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")
print(f"\nUnique hexagons : {N_HEX}  |  embed dim : {HEX_EMBED_DIM}")


Features (29): ['is_weekend', 'is_rush_hour', 'is_holiday', 'hour_of_day_sin', 'hour_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'temperature_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'wind_speed_10m', 'cloud_cover', 'is_day', 'area_km2', 'dist_to_nearest_airport_km', 'dist_to_nearest_train_station_km', 'dist_to_nearest_stadium_km', 'train_station_per_km2', 'restaurants_per_km2', 'bars_and_clubs_per_km2', 'hotels_per_km2', 'hospitals_per_km2', 'universities_per_km2', 'attractions_per_km2', 'poi_density_total_per_km2']

X_train : (981120, 29)   y_train : (981120,)
X_val   : (210240, 29)     y_val   : (210240,)
X_test  : (210240, 29)    y_test  : (210240,)

Unique hexagons : 160  |  embed dim : 16


## LR Search — Baseline Model

Each architecture gets its own LR search; the winner is stored in `BEST_LR[arch]`.
Training and evaluation cells always read from `BEST_LR`, falling back to
`config.LEARNING_RATE` if a search has not been run for that arch yet.


In [ ]:
run_lr_search(ARCH_BASELINE, X_train, y_train, X_val, y_val, hex_train, hex_val, device=device)  # (2, 64)


## Baseline Model — Training

Run `ARCH_BASELINE = (2 hidden layers, 64 units)` across all seeds and report
the mean ± std validation loss.


In [54]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Visualization          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

input_dim = X_train.shape[1]
n_layers, width = ARCH_BASELINE

viz_model = DemandBaseline(input_dim, n_layers, width, N_HEX, HEX_EMBED_DIM)
x_dummy   = torch.zeros(1, input_dim)
h_dummy   = torch.zeros(1, dtype=torch.long)
summary(viz_model, input_data=(x_dummy, h_dummy), col_names=["input_size", "output_size", "num_params"], verbose=1)


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
DemandBaseline                           [1, 26]                   [1]                       --
├─Sequential: 1-1                        [1, 26]                   [1, 1]                    --
│    └─Linear: 2-1                       [1, 26]                   [1, 64]                   1,728
│    └─ReLU: 2-2                         [1, 64]                   [1, 64]                   --
│    └─Linear: 2-3                       [1, 64]                   [1, 64]                   4,160
│    └─ReLU: 2-4                         [1, 64]                   [1, 64]                   --
│    └─Linear: 2-5                       [1, 64]                   [1, 1]                    65
│    └─Softplus: 2-6                     [1, 1]                    [1, 1]                    --
Total params: 5,953
Trainable params: 5,953
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.01
Input size (MB): 

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
DemandBaseline                           [1, 26]                   [1]                       --
├─Sequential: 1-1                        [1, 26]                   [1, 1]                    --
│    └─Linear: 2-1                       [1, 26]                   [1, 64]                   1,728
│    └─ReLU: 2-2                         [1, 64]                   [1, 64]                   --
│    └─Linear: 2-3                       [1, 64]                   [1, 64]                   4,160
│    └─ReLU: 2-4                         [1, 64]                   [1, 64]                   --
│    └─Linear: 2-5                       [1, 64]                   [1, 1]                    65
│    └─Softplus: 2-6                     [1, 1]                    [1, 1]                    --
Total params: 5,953
Trainable params: 5,953
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.01
Input size (MB): 

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Training               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

print(f"Device: {device}\n")

baseline_val_losses  = []
baseline_models      = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_BASELINE, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_BASELINE, config.LEARNING_RATE),
    )
    baseline_val_losses.append(val_loss)
    baseline_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nBaseline  val loss:  {np.mean(baseline_val_losses):.4f} ± {np.std(baseline_val_losses):.4f}")


## Baseline Model — Evaluation

Evaluate on the held-out **test set** using the best-checkpoint model from each
seed, then report mean ± std across seeds.

Metrics:
- **MAE** — mean absolute error (interpretable in trip counts)
- **RMSE** — root mean squared error (penalises large misses more)
- **MAPE** — mean absolute percentage error (relative, skip zeros)


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Evaluation             #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results = []
mae_scores, rmse_scores, mape_scores = [], [], []
n_layers, width = ARCH_BASELINE

for seed, model in zip(SEEDS, baseline_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t, hex_test_t).cpu().numpy()

    mae  = np.mean(np.abs(preds - y_test))
    rmse = np.sqrt(np.mean((preds - y_test) ** 2))
    mask = y_test > 0
    mape = np.mean(np.abs((preds[mask] - y_test[mask]) / y_test[mask])) * 100

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    mape_scores.append(mape)
    print(f"  seed={seed}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

    run_results.append({
        "timestamp"     : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"         : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"      : n_layers,
        "width"         : width,
        "learning_rate" : BEST_LR.get(ARCH_BASELINE, config.LEARNING_RATE),
        "batch_size"    : BATCH_SIZE,
        "seed"          : seed,
        "val_loss"      : baseline_val_losses[seed],
        "mae"           : round(mae,  6),
        "rmse"          : round(rmse, 6),
        "mape"          : round(mape, 4),
    })

print(f"\nBaseline  MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"Baseline  RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"Baseline  MAPE : {np.mean(mape_scores):.2f}% ± {np.std(mape_scores):.2f}%")


In [35]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


Created data/results/nn_results.csv with 5 rows
          timestamp    model  n_layers  width  learning_rate  batch_size  seed  val_loss      mae     rmse     mape
2026-06-13T14:44:03 baseline         2     64          0.005         256     0 -0.154457 0.522698 3.936998 148.9188
2026-06-13T14:44:03 baseline         2     64          0.005         256     1 -0.141265 0.565017 3.945495 167.6884
2026-06-13T14:44:03 baseline         2     64          0.005         256     2 -0.141709 0.578841 3.956541 166.8561
2026-06-13T14:44:03 baseline         2     64          0.005         256     3 -0.156943 0.545149 3.939008 169.7217
2026-06-13T14:44:04 baseline         2     64          0.005         256     4 -0.147501 0.611058 3.977664 217.3335


## LR Search — Deeper Model

Same grid search as for the baseline, run on `ARCH_DEEPER = (4 hidden layers, 64 units)`.
Result is stored in `BEST_LR[ARCH_DEEPER]`.


In [ ]:
run_lr_search(ARCH_DEEPER,   X_train, y_train, X_val, y_val, hex_train, hex_val, device=device)  # (4, 64)


## Deeper Model — Training

Run `ARCH_DEEPER = (4 hidden layers, 64 units)` across all seeds.
Width is fixed; depth doubles relative to the baseline.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Deeper Model — Training                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

deeper_val_losses = []
deeper_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_DEEPER, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_DEEPER, config.LEARNING_RATE),
    )
    deeper_val_losses.append(val_loss)
    deeper_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nDeeper  val loss:  {np.mean(deeper_val_losses):.4f} ± {np.std(deeper_val_losses):.4f}")


## Deeper Model — Evaluation


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Deeper Model — Evaluation               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results = []
mae_scores, rmse_scores, mape_scores = [], [], []
n_layers, width = ARCH_DEEPER

for seed, model in zip(SEEDS, deeper_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t, hex_test_t).cpu().numpy()

    mae  = np.mean(np.abs(preds - y_test))
    rmse = np.sqrt(np.mean((preds - y_test) ** 2))
    mask = y_test > 0
    mape = np.mean(np.abs((preds[mask] - y_test[mask]) / y_test[mask])) * 100

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    mape_scores.append(mape)
    print(f"  seed={seed}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

    run_results.append({
        "timestamp"     : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"         : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"      : n_layers,
        "width"         : width,
        "learning_rate" : BEST_LR.get(ARCH_DEEPER, config.LEARNING_RATE),
        "batch_size"    : BATCH_SIZE,
        "seed"          : seed,
        "val_loss"      : deeper_val_losses[seed],
        "mae"           : round(mae,  6),
        "rmse"          : round(rmse, 6),
        "mape"          : round(mape, 4),
    })

print(f"\nDeeper  MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"Deeper  RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"Deeper  MAPE : {np.mean(mape_scores):.2f}% ± {np.std(mape_scores):.2f}%")


In [40]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


Appended 5 rows to data/results/nn_results.csv
          timestamp model  n_layers  width  learning_rate  batch_size  seed  val_loss      mae     rmse     mape
2026-06-14T12:43:12  deep         4     64        0.00001         256     0 -0.087047 0.933722 4.177202 292.4367
2026-06-14T12:43:12  deep         4     64        0.00001         256     1 -0.056223 1.008265 4.253420 335.9738
2026-06-14T12:43:12  deep         4     64        0.00001         256     2 -0.071655 0.881631 4.183690 254.8624
2026-06-14T12:43:12  deep         4     64        0.00001         256     3 -0.038341 1.018100 4.254317 307.5181
2026-06-14T12:43:13  deep         4     64        0.00001         256     4 -0.079930 1.027786 4.371970 349.3288


## LR Search — Wider Model

Same grid search as for the baseline, run on `ARCH_WIDER = (2 hidden layers, 128 units)`.
Result is stored in `BEST_LR[ARCH_WIDER]`.


In [ ]:
run_lr_search(ARCH_WIDER,    X_train, y_train, X_val, y_val, hex_train, hex_val, device=device)  # (2, 128)


## Wider Model — Training

Run `ARCH_WIDER = (2 hidden layers, 128 units)` across all seeds.
Depth is fixed; width doubles relative to the baseline.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Wider Model — Training                  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

wider_val_losses = []
wider_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_WIDER, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_WIDER, config.LEARNING_RATE),
    )
    wider_val_losses.append(val_loss)
    wider_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nWider  val loss:  {np.mean(wider_val_losses):.4f} ± {np.std(wider_val_losses):.4f}")


## Wider Model — Evaluation


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Wider Model — Evaluation                #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results = []
mae_scores, rmse_scores, mape_scores = [], [], []
n_layers, width = ARCH_WIDER

for seed, model in zip(SEEDS, wider_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t, hex_test_t).cpu().numpy()

    mae  = np.mean(np.abs(preds - y_test))
    rmse = np.sqrt(np.mean((preds - y_test) ** 2))
    mask = y_test > 0
    mape = np.mean(np.abs((preds[mask] - y_test[mask]) / y_test[mask])) * 100

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    mape_scores.append(mape)
    print(f"  seed={seed}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

    run_results.append({
        "timestamp"     : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"         : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"      : n_layers,
        "width"         : width,
        "learning_rate" : BEST_LR.get(ARCH_WIDER, config.LEARNING_RATE),
        "batch_size"    : BATCH_SIZE,
        "seed"          : seed,
        "val_loss"      : wider_val_losses[seed],
        "mae"           : round(mae,  6),
        "rmse"          : round(rmse, 6),
        "mape"          : round(mape, 4),
    })

print(f"\nWider  MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"Wider  RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"Wider  MAPE : {np.mean(mape_scores):.2f}% ± {np.std(mape_scores):.2f}%")


In [48]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


Appended 5 rows to data/results/nn_results.csv
          timestamp model  n_layers  width  learning_rate  batch_size  seed  val_loss      mae     rmse     mape
2026-06-14T15:16:19  wide         2    128        0.00001         256     0 -0.087856 1.024753 4.221322 325.5545
2026-06-14T15:16:20  wide         2    128        0.00001         256     1 -0.068899 0.906256 4.112021 250.5003
2026-06-14T15:16:20  wide         2    128        0.00001         256     2 -0.066387 0.907339 4.112060 255.3093
2026-06-14T15:16:20  wide         2    128        0.00001         256     3 -0.088028 1.129276 4.336548 339.8401
2026-06-14T15:16:21  wide         2    128        0.00001         256     4 -0.078890 1.142359 4.330955 360.8458
